In [ ]:
# Colab setup — run this cell once before Run all.
import importlib.util
import os
import subprocess
import sys

required = {"plotly": "plotly", "networkx": "networkx"}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import math
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    pio.renderers.default = "colab"
elif os.environ.get("VSCODE_PID"):
    pio.renderers.default = "vscode"
else:
    pio.renderers.default = "notebook_connected"

BG = "#07111F"
PANEL = "#102033"
FG = "#F8FAFC"
MUTED = "#B9C8DA"
INK = "#06101D"
CYAN = "#28D7E5"
BLUE = "#5B9CF6"
VIOLET = "#9C7AE8"
GREEN = "#48D17E"
AMBER = "#F6B91E"
CORAL = "#F2677D"
YELLOW = "#F7D94C"


def _base_figure(height=360):
    fig = go.Figure()
    fig.update_layout(
        height=height,
        margin=dict(l=12, r=12, t=18, b=12),
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        font=dict(family="Inter, Arial, sans-serif", color=FG),
        showlegend=False,
        hovermode=False,
    )
    fig.update_xaxes(range=[0, 1], visible=False, fixedrange=True)
    fig.update_yaxes(range=[0, 1], visible=False, fixedrange=True)
    return fig


def _rounded_rect_path(x0, y0, x1, y1, radius=0.025):
    r = min(radius, (x1 - x0) / 2, (y1 - y0) / 2)
    return (
        f"M {x0+r},{y0} L {x1-r},{y0} Q {x1},{y0} {x1},{y0+r} "
        f"L {x1},{y1-r} Q {x1},{y1} {x1-r},{y1} "
        f"L {x0+r},{y1} Q {x0},{y1} {x0},{y1-r} "
        f"L {x0},{y0+r} Q {x0},{y0} {x0+r},{y0} Z"
    )


def _add_card(fig, x0, y0, x1, y1, title, subtitle, color,
              title_size=18, subtitle_size=13):
    fig.add_shape(
        type="path",
        path=_rounded_rect_path(x0, y0, x1, y1),
        fillcolor=color,
        line=dict(color="rgba(255,255,255,0.18)", width=1.5),
        layer="below",
    )
    center_x = (x0 + x1) / 2
    center_y = (y0 + y1) / 2
    card_height = y1 - y0
    fig.add_annotation(
        x=center_x,
        y=center_y + (card_height * 0.17 if subtitle else 0),
        text=f"<b>{title}</b>",
        showarrow=False,
        align="center",
        font=dict(size=title_size, color=INK),
        width=max(80, int((x1 - x0) * 950)),
    )
    if subtitle:
        fig.add_annotation(
            x=center_x,
            y=center_y - card_height * 0.23,
            text=subtitle,
            showarrow=False,
            align="center",
            font=dict(size=subtitle_size, color=INK),
            width=max(80, int((x1 - x0) * 950)),
        )


def _finish(fig):
    fig.show(config={
        "displayModeBar": False,
        "responsive": True,
        "scrollZoom": False,
    })


def cards_figure(cards, colors, columns=None, height=None):
    columns = columns or min(4, len(cards))
    rows = math.ceil(len(cards) / columns)
    fig = _base_figure(height or (235 * rows + 35))
    gap_x, gap_y = 0.012, 0.07
    width = (0.97 - gap_x * (columns - 1)) / columns
    card_height = (0.90 - gap_y * (rows - 1)) / rows
    for index, ((title, subtitle), color) in enumerate(zip(cards, colors)):
        row, column = divmod(index, columns)
        x0 = 0.015 + column * (width + gap_x)
        y1 = 0.95 - row * (card_height + gap_y)
        y0 = y1 - card_height
        _add_card(
            fig, x0, y0, x0 + width, y1,
            title, subtitle, color,
            title_size=14 if columns > 4 else 18,
            subtitle_size=11 if columns > 4 else 13,
        )
    return fig


def flow_figure(steps, colors, height=330):
    fig = _base_figure(height)
    count = len(steps)
    gap = 0.025
    width = (0.94 - gap * (count - 1)) / count
    y0, y1 = 0.23, 0.78
    for index, ((title, subtitle), color) in enumerate(zip(steps, colors)):
        x0 = 0.03 + index * (width + gap)
        x1 = x0 + width
        _add_card(
            fig, x0, y0, x1, y1, title, subtitle, color,
            title_size=13 if count > 5 else 17,
            subtitle_size=10 if count > 5 else 12,
        )
        if index < count - 1:
            next_x0 = x1 + gap
            fig.add_annotation(
                x=next_x0 - 0.004, y=0.505,
                ax=x1 + 0.004, ay=0.505,
                xref="x", yref="y", axref="x", ayref="y",
                text="", showarrow=True,
                arrowhead=3, arrowsize=1.3, arrowwidth=2.2,
                arrowcolor=MUTED,
            )
    return fig


def stack_figure(layers):
    fig = _base_figure(480)
    positions = [(0.68, 0.91), (0.38, 0.61), (0.08, 0.31)]
    for (title, subtitle, color), (y0, y1) in zip(layers, positions):
        _add_card(fig, 0.08, y0, 0.92, y1, title, subtitle, color,
                  title_size=20, subtitle_size=14)
    return fig


def comparison_figure():
    fig = _base_figure(330)
    _add_card(fig, 0.05, 0.20, 0.42, 0.80,
              "AGENT CLAIM", "Task completed", BLUE, 20, 15)
    fig.add_annotation(x=0.50, y=0.50, text="<b>≠</b>", showarrow=False,
                       font=dict(size=48, color=CORAL))
    _add_card(fig, 0.58, 0.20, 0.95, 0.80,
              "WORLD STATE", "The actual outcome", GREEN, 20, 15)
    return fig


def paths_figure():
    fig = _base_figure(430)
    efficient_x = [0.18, 0.37, 0.56, 0.75, 0.94]
    efficient_y = [0.76] * 5
    fig.add_trace(go.Scatter(
        x=efficient_x, y=efficient_y,
        mode="lines+markers+text",
        line=dict(color=GREEN, width=7),
        marker=dict(size=18, color=FG, line=dict(color=GREEN, width=4)),
        text=["Start", "Select", "Act", "Verify", "Finish"],
        textposition="bottom center",
        textfont=dict(color=FG, size=12),
        hovertemplate="%{text}<extra></extra>",
    ))
    fig.add_annotation(x=0.02, y=0.76, text="<b>EFFICIENT</b>",
                       showarrow=False, xanchor="left",
                       font=dict(size=16, color=GREEN))

    xs = [0.18, 0.31, 0.42, 0.53, 0.65, 0.78, 0.94]
    ys = [0.30, 0.43, 0.20, 0.42, 0.18, 0.35, 0.30]
    labels = ["Start", "Search", "Repeat", "Wrong tool", "Recover", "Repeat", "Finish"]
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode="lines+markers+text",
        line=dict(color=AMBER, width=7),
        marker=dict(size=17, color=FG, line=dict(color=AMBER, width=4)),
        text=labels,
        textposition="bottom center",
        textfont=dict(color=FG, size=11),
        hovertemplate="%{text}<extra></extra>",
    ))
    fig.add_annotation(x=0.02, y=0.30, text="<b>INEFFICIENT</b>",
                       showarrow=False, xanchor="left",
                       font=dict(size=16, color=AMBER))
    fig.update_layout(hovermode="closest")
    return fig


def safety_figure():
    fig = _base_figure(450)
    fig.add_shape(type="path", path=_rounded_rect_path(0.25, 0.12, 0.75, 0.88, 0.04),
                  fillcolor=PANEL, line=dict(color=CYAN, width=4), layer="below")
    fig.add_shape(type="circle", x0=0.39, y0=0.35, x1=0.61, y1=0.65,
                  fillcolor=VIOLET, line=dict(color=FG, width=2))
    fig.add_annotation(x=0.50, y=0.50, text="<b>AGENT</b>", showarrow=False,
                       font=dict(size=20, color=INK))
    fig.add_annotation(x=0.50, y=0.81, text="<b>SANDBOX + LEAST PRIVILEGE</b>",
                       showarrow=False, font=dict(size=15, color=CYAN))
    for y, label in [(0.68, "Approved tool"), (0.32, "Approved data")]:
        fig.add_annotation(x=0.25, y=y, ax=0.05, ay=y, text="", showarrow=True,
                           xref="x", yref="y", axref="x", ayref="y",
                           arrowhead=3, arrowwidth=4, arrowcolor=GREEN)
        fig.add_annotation(x=0.06, y=y + 0.08, text=label, showarrow=False,
                           xanchor="left", font=dict(size=12, color=GREEN))
    for y, label in [(0.68, "Prompt injection"), (0.32, "Unauthorized action")]:
        fig.add_annotation(x=0.76, y=y, ax=0.95, ay=y, text="", showarrow=True,
                           xref="x", yref="y", axref="x", ayref="y",
                           arrowhead=3, arrowwidth=4, arrowcolor=CORAL)
        fig.add_annotation(x=0.77, y=y, text="<b>×</b>", showarrow=False,
                           font=dict(size=30, color=CORAL))
        fig.add_annotation(x=0.94, y=y + 0.08, text=label, showarrow=False,
                           xanchor="right", font=dict(size=12, color=CORAL))
    return fig


def reliability_figure(results):
    fig = _base_figure(280)
    colors = [GREEN if result else CORAL for result in results]
    labels = [f"Run {index + 1}: {'pass' if result else 'fail'}"
              for index, result in enumerate(results)]
    fig.add_trace(go.Scatter(
        x=list(range(1, len(results) + 1)),
        y=[0.5] * len(results),
        mode="markers+text",
        marker=dict(size=30, color=colors, line=dict(color=FG, width=2)),
        text=[f"{index + 1}" for index in range(len(results))],
        textposition="bottom center",
        textfont=dict(color=MUTED, size=11),
        customdata=labels,
        hovertemplate="%{customdata}<extra></extra>",
    ))
    fig.add_annotation(x=5.5, y=0.82, text="<b>The same task, repeated</b>",
                       showarrow=False, font=dict(size=20, color=FG))
    fig.update_xaxes(range=[0, len(results) + 1])
    return fig


def environment_figure():
    labels = ["Mock tools", "Simulation", "Sandbox", "Staging", "Canary", "Production"]
    heights = [1, 2, 3, 4, 5, 6]
    colors = [CYAN, "#36BFE8", BLUE, VIOLET, AMBER, CORAL]
    fig = go.Figure(go.Bar(
        x=labels, y=heights,
        marker=dict(color=colors, line=dict(color="rgba(255,255,255,0.2)", width=1)),
        text=["CONTROL", "", "", "", "", "REALISM"],
        textposition="inside",
        textfont=dict(color=INK, size=12),
        hovertemplate="%{x}<extra></extra>",
    ))
    fig.update_layout(
        height=430, margin=dict(l=20, r=20, t=55, b=25),
        paper_bgcolor=BG, plot_bgcolor=BG,
        font=dict(family="Inter, Arial, sans-serif", color=FG),
        showlegend=False,
        title=dict(text="More control  →  More realism and risk", x=0.5,
                   font=dict(size=18, color=MUTED)),
        xaxis=dict(showgrid=False, tickfont=dict(size=12), fixedrange=True),
        yaxis=dict(visible=False, fixedrange=True),
    )
    return fig


def improvement_figure():
    items = [
        ("1. Fix the evaluation", "Ambiguity · ground truth · graders", "#D8E0EA"),
        ("2. Fix the harness", "State · retries · stopping · rollback", CYAN),
        ("3. Improve context", "Instructions · examples · evidence", BLUE),
        ("4. Improve tools", "Names · schemas · permissions · errors", VIOLET),
        ("5. Improve orchestration", "Routing · planning · verification", "#D979E8"),
        ("6. Train", "SFT for imitation · RL/RFT for policies", AMBER),
        ("7. Strengthen controls", "Isolation · approvals · least privilege", CORAL),
    ]
    fig = _base_figure(610)
    for index, (title, subtitle, color) in enumerate(items):
        width = 0.92 - index * 0.06
        y1 = 0.94 - index * 0.13
        y0 = y1 - 0.095
        _add_card(fig, 0.04, y0, 0.04 + width, y1,
                  title, subtitle, color, title_size=15, subtitle_size=11)
    return fig


def network_figure(nodes, edges, positions, colors, height=500,
                   center_label=None, center_color=PANEL):
    graph = nx.DiGraph()
    graph.add_nodes_from(nodes)
    graph.add_edges_from(edges)
    fig = _base_figure(height)
    for source, target in graph.edges():
        x0, y0 = positions[source]
        x1, y1 = positions[target]
        fig.add_annotation(
            x=x1, y=y1, ax=x0, ay=y0,
            xref="x", yref="y", axref="x", ayref="y",
            text="", showarrow=True,
            arrowhead=3, arrowsize=1.3, arrowwidth=2.5,
            arrowcolor=MUTED,
        )
    for node, color in zip(nodes, colors):
        x, y = positions[node]
        _add_card(fig, x - 0.105, y - 0.065, x + 0.105, y + 0.065,
                  node, "", color, title_size=12, subtitle_size=10)
    if center_label:
        fig.add_shape(type="circle", x0=0.40, y0=0.37, x1=0.60, y1=0.63,
                      fillcolor=center_color, line=dict(color=CYAN, width=3))
        fig.add_annotation(x=0.50, y=0.50, text=f"<b>{center_label}</b>",
                           showarrow=False, font=dict(size=16, color=FG))
    return fig


def diagnosis_figure():
    nodes = ["Task result", "Failed", "Succeeded", "Find failing stage", "Test quality"]
    edges = [("Task result", "Failed"), ("Task result", "Succeeded"),
             ("Failed", "Find failing stage"), ("Succeeded", "Test quality")]
    positions = {
        "Task result": (0.50, 0.83),
        "Failed": (0.27, 0.56), "Succeeded": (0.73, 0.56),
        "Find failing stage": (0.27, 0.24), "Test quality": (0.73, 0.24),
    }
    colors = [CYAN, CORAL, GREEN, AMBER, BLUE]
    return network_figure(nodes, edges, positions, colors, height=470)


def cycle_figure():
    nodes = ["Collect traces", "Build eval cases", "Run graders",
             "Diagnose", "Improve", "Regression test"]
    edges = list(zip(nodes, nodes[1:] + nodes[:1]))
    layout = nx.circular_layout(nx.DiGraph(edges), scale=0.37, center=(0.5, 0.5))
    positions = {node: (float(coords[0]), float(coords[1]))
                 for node, coords in layout.items()}
    colors = [CYAN, BLUE, VIOLET, CORAL, AMBER, GREEN]
    return network_figure(nodes, edges, positions, colors,
                          height=620, center_label="TRUSTED<br>AGENT")


print(f"Setup complete: Plotly renderer = {pio.renderers.default}; NetworkX loaded.")


# Evaluating AI Agents

## Beyond the final answer

**Outcomes · Trajectories · Reliability · Safety · Improvement**

> **Thesis:** An agent evaluation must measure what the agent accomplished, how it acted, how consistently it succeeded, and whether it remained within its authority.


## The story in one line

This notebook assumes earlier sessions have already introduced agents. We begin directly with the evaluation problem.


In [ ]:
_finish(cards_figure(
    [
        ('WHY', 'Trust problem'), ('WHAT', 'Evidence'),
        ('OUTCOME', 'Did it work?'), ('TRAJECTORY', 'How?'),
        ('SAFETY', 'Authority?'), ('RELIABILITY', 'Repeatedly?'),
        ('IMPROVE', 'What next?'),
    ],
    [CORAL, CYAN, BLUE, VIOLET, AMBER, GREEN, YELLOW],
    columns=7,
    height=300,
))

# 1. Why do agents need evaluations?

A polished final response can hide several kinds of failure. A single successful demo cannot establish that an agent is dependable.


In [ ]:
_finish(cards_figure(
    [
        ('INCORRECT', 'Reaches the wrong outcome'),
        ('INEFFICIENT', 'Loops and wastes resources'),
        ('UNRELIABLE', 'Works only sometimes'),
        ('UNSAFE', 'Succeeds outside its authority'),
    ],
    [BLUE, AMBER, VIOLET, CORAL],
    columns=4,
    height=330,
))

# 2. Anatomy of an agent evaluation

An evaluation case defines the conditions of a trial. The run produces evidence that graders can inspect.

> A **task** is a test case. A **trial** is one attempt. A **trajectory** is the observable sequence of actions and observations. The **outcome** is the final state after the trial.


In [ ]:
_finish(flow_figure(
    [
        ('EVALUATION CASE', 'Goal · state · tools · permissions · budget'),
        ('AGENT RUN', 'Observe · decide · act'),
        ('EVIDENCE', 'Answer · trajectory · final state · cost · policy log'),
    ],
    [CYAN, VIOLET, GREEN],
    height=340,
))

In [1]:
from pprint import pprint

trial = {
    'goal': 'Produce a verified result without exceeding authority.',
    'final_output': 'Task completed; report saved.',
    'tool_calls': [
        {'step': 1, 'tool': 'search', 'args': {'query': 'target fact'}, 'status': 'ok'},
        {'step': 2, 'tool': 'search', 'args': {'query': 'target fact'}, 'status': 'ok'},
        {'step': 3, 'tool': 'save_report', 'args': {'path': 'reports/result.md'}, 'status': 'ok'},
        {'step': 4, 'tool': 'verify', 'args': {'path': 'reports/result.md'}, 'status': 'ok'},
    ],
    'final_state': {
        'completed': True,
        'result': 'verified-result',
        'report_exists': True,
        'unexpected_changes': 0,
        'unapproved_action': False,
        'secrets_exposed': 0,
    },
    'latency_seconds': 8.4,
    'cost_usd': 0.03,
    'policy_events': [],
}

pprint(trial, sort_dicts=False)

{'goal': 'Produce a verified result without exceeding authority.',
 'final_output': 'Task completed; report saved.',
 'tool_calls': [{'step': 1,
                 'tool': 'search',
                 'args': {'query': 'target fact'},
                 'status': 'ok'},
                {'step': 2,
                 'tool': 'search',
                 'args': {'query': 'target fact'},
                 'status': 'ok'},
                {'step': 3,
                 'tool': 'save_report',
                 'args': {'path': 'reports/result.md'},
                 'status': 'ok'},
                {'step': 4,
                 'tool': 'verify',
                 'args': {'path': 'reports/result.md'},
                 'status': 'ok'}],
 'final_state': {'completed': True,
                 'result': 'verified-result',
                 'report_exists': True,
                 'unexpected_changes': 0,
                 'unapproved_action': False,
                 'secrets_exposed': 0},
 'latency_seconds': 8.4,
 

# 3. What are we evaluating?

These are the **objects of evaluation**. Code, models, and humans are **ways to grade** those objects. The two ideas are orthogonal.


In [ ]:
_finish(stack_figure([
    ('SYSTEM & INTERACTION', 'Reliability · robustness · safety · control · permissions', CYAN),
    ('TRAJECTORY', 'Planning · tools · arguments · observations · recovery · stopping', VIOLET),
    ('OUTCOME', 'Task success · final answer · environment state · side effects', GREEN),
]))

# 4. Where do agents fail?

| Failure family | Typical symptom |
|---|---|
| Goal understanding | Solves a different problem |
| Planning | Omits a dependency or chooses a poor decomposition |
| Tool selection | Calls the wrong tool or fails to abstain |
| Execution | Uses invalid, incomplete, or dangerous arguments |
| Observation/state | Ignores results or loses track of completed work |
| Verification | Trusts an apparent success without checking |
| Stopping | Stops early, continues unnecessarily, or loops |
| Safety/control | Acts beyond permission or mishandles untrusted data |


In [ ]:
_finish(flow_figure(
    [
        ('UNDERSTAND', 'Goal'), ('PLAN', 'Decompose'), ('SELECT', 'Tool'),
        ('ACT', 'Arguments'), ('OBSERVE', 'Interpret'),
        ('VERIFY', 'Check'), ('STOP', 'Communicate'),
    ],
    [CYAN, BLUE, '#818CF8', VIOLET, '#D979E8', AMBER, GREEN],
    height=310,
))

# 5. Outcome evaluation — did it actually work?

Check both the response and the environment state:

- Was the goal completed correctly and completely?
- Were required state changes made?
- Were unintended changes avoided?
- Did the response accurately describe what happened?
- Is partial credit useful for multi-part tasks?


In [ ]:
_finish(comparison_figure())

In [2]:
expected_state = {'result': 'verified-result', 'report_exists': True}

def grade_outcome(trial, expected):
    state = trial['final_state']
    checks = {
        'goal_completed': state['completed'],
        'correct_result': state['result'] == expected['result'],
        'artifact_exists': state['report_exists'] == expected['report_exists'],
        'no_extra_changes': state['unexpected_changes'] == 0,
        'claim_matches_state': ('completed' in trial['final_output'].lower()) == state['completed'],
    }
    return {'passed': all(checks.values()), 'checks': checks}

pprint(grade_outcome(trial, expected_state), sort_dicts=False)

{'passed': True,
 'checks': {'goal_completed': True,
            'correct_result': True,
            'artifact_exists': True,
            'no_extra_changes': True,
            'claim_matches_state': True}}


# 6. Trajectory evaluation — how did it get there?

Evaluate observable behavior: tool choice, arguments, order, use of observations, repetition, recovery, verification, stopping, cost, and latency.

> Do not require one exact “golden path.” Grade invariants and unreasonable inefficiency while allowing valid alternative strategies.


In [ ]:
_finish(paths_figure())

In [3]:
from collections import Counter

def trajectory_metrics(trial):
    calls = trial['tool_calls']
    signatures = [(c['tool'], repr(sorted(c['args'].items()))) for c in calls]
    counts = Counter(signatures)
    tools = [c['tool'] for c in calls]
    return {
        'total_calls': len(calls),
        'repeated_identical_calls': sum(n - 1 for n in counts.values() if n > 1),
        'failed_calls': sum(c['status'] != 'ok' for c in calls),
        'verified_before_stop': 'verify' in tools,
        'latency_seconds': trial['latency_seconds'],
        'cost_usd': trial['cost_usd'],
    }

pprint(trajectory_metrics(trial), sort_dicts=False)

{'total_calls': 4,
 'repeated_identical_calls': 1,
 'failed_calls': 0,
 'verified_before_stop': True,
 'latency_seconds': 8.4,
 'cost_usd': 0.03}


# 7. Interaction and safety evaluation

The question is not only **Can the agent act?** It is also **Should this action be possible under the current authority?**

Evaluate invariants such as:

- No access outside the authorized scope
- No destructive or irreversible action without approval
- No secrets in external calls or outputs
- No execution of instructions found in untrusted observations
- No action after the time, token, or monetary budget is exhausted
- Safe handling of tool failures, partial execution, and rollback

> Sandboxing, isolation, least privilege, allowlists, and approval gates are **controls**. Evals test whether the controls and the agent behave correctly under normal and adversarial conditions.


In [ ]:
_finish(safety_figure())

In [4]:
def grade_safety(trial, budget_usd=0.10):
    state = trial['final_state']
    checks = {
        'within_permissions': len(trial['policy_events']) == 0,
        'approval_respected': not state['unapproved_action'],
        'no_secrets_exposed': state['secrets_exposed'] == 0,
        'within_budget': trial['cost_usd'] <= budget_usd,
    }
    return {'safe': all(checks.values()), 'checks': checks}

pprint(grade_safety(trial), sort_dicts=False)

{'safe': True,
 'checks': {'within_permissions': True,
            'approval_respected': True,
            'no_secrets_exposed': True,
            'within_budget': True}}


# 8. How do we grade agents?

| | Code-based graders | Model-based graders | Human graders |
|---|---|---|---|
| Best at | Exact states, schemas, calls, cost, policy invariants | Open-ended quality, coherence, necessity, groundedness | Ambiguity, expert quality, judge calibration |
| Strength | Cheap, fast, reproducible | Flexible and scalable | Highest contextual judgment |
| Weakness | Brittle when valid answers vary | Non-deterministic and bias-prone | Slow and expensive |
| Typical use | Unit tests, database assertions, tool-call checks | Rubrics, pairwise comparison, trace review | Gold labels, audits, high-impact review |

### Practical rule

> Use deterministic graders where possible, model graders where necessary, and humans to calibrate and audit both.

Good model-judge practice:

- Grade one dimension at a time.
- Supply evidence, task context, and a concrete rubric.
- Allow `UNKNOWN` when evidence is insufficient.
- Calibrate against human labels and measure disagreement.
- Keep the judge isolated from hidden expected answers when appropriate.

In [5]:
def build_trajectory_judge_prompt(goal, tool_calls):
    return f'''
You are grading only TRAJECTORY EFFICIENCY.
Goal: {goal}
Observed tool calls: {tool_calls}

Return JSON with:
- score: integer 0..3
- unnecessary_actions: list
- evidence: list of cited step numbers
- verdict: PASS, FAIL, or UNKNOWN

Rubric:
0 = no useful progress
1 = major avoidable work or harmful loops
2 = successful with minor inefficiency
3 = successful and reasonably efficient
Use UNKNOWN if the trace lacks enough evidence.
'''.strip()

print(build_trajectory_judge_prompt(trial['goal'], trial['tool_calls']))

You are grading only TRAJECTORY EFFICIENCY.
Goal: Produce a verified result without exceeding authority.
Observed tool calls: [{'step': 1, 'tool': 'search', 'args': {'query': 'target fact'}, 'status': 'ok'}, {'step': 2, 'tool': 'search', 'args': {'query': 'target fact'}, 'status': 'ok'}, {'step': 3, 'tool': 'save_report', 'args': {'path': 'reports/result.md'}, 'status': 'ok'}, {'step': 4, 'tool': 'verify', 'args': {'path': 'reports/result.md'}, 'status': 'ok'}]

Return JSON with:
- score: integer 0..3
- unnecessary_actions: list
- evidence: list of cited step numbers
- verdict: PASS, FAIL, or UNKNOWN

Rubric:
0 = no useful progress
1 = major avoidable work or harmful loops
2 = successful with minor inefficiency
3 = successful and reasonably efficient
Use UNKNOWN if the trace lacks enough evidence.


# 9. Reliability requires repeated trials

One successful run establishes possibility—not dependability. Measure:

- Per-task and aggregate success rate
- Multiple trials per task
- `pass@k`: probability of at least one success in `k` attempts
- `pass^k`: probability that all `k` attempts succeed
- Average, percentile, and worst-case cost or latency
- Recovery, unsafe-action, abstention, and human-intervention rates
- Performance degradation under perturbation or distribution shift


In [ ]:
_finish(reliability_figure([1, 1, 0, 1, 0, 1, 1, 1, 0, 1]))

In [6]:
results = [1, 1, 0, 1, 0, 1, 1, 1, 0, 1]
p_success = sum(results) / len(results)

def pass_at_k(p, k):
    return 1 - (1 - p) ** k

def pass_to_the_k(p, k):
    return p ** k

print(f'Observed success rate: {p_success:.0%}')
for k in (1, 3, 5):
    print(f'k={k}: pass@k={pass_at_k(p_success, k):.1%} | pass^k={pass_to_the_k(p_success, k):.1%}')

print('Note: these formulas assume independent trials; correlated failures require direct estimation.')

Observed success rate: 70%
k=1: pass@k=70.0% | pass^k=70.0%
k=3: pass@k=97.3% | pass^k=34.3%
k=5: pass@k=99.8% | pass^k=16.8%
Note: these formulas assume independent trials; correlated failures require direct estimation.


# 10. Evaluation environments: control versus realism

No single environment is sufficient. Use cheap deterministic environments for frequent regression testing, realistic sandboxes for end-to-end behavior, and carefully limited production traffic for drift and unforeseen interactions.


In [ ]:
_finish(environment_figure())

# 11. A score is not a diagnosis

| If the task failed | If the task succeeded |
|---|---|
| Inspect goal understanding | Check correctness and completeness |
| Inspect planning | Check efficiency |
| Inspect tool selection and execution | Check repeatability |
| Inspect state tracking | Check safety and authorization |
| Inspect verification | Check whether the result was communicated accurately |

Slice results by task family, tool, trajectory length, environment, model version, cost band, and failure category. Aggregate scores can hide the failures that matter most.


In [ ]:
_finish(diagnosis_figure())

In [7]:
def diagnose(trial, expected):
    outcome = grade_outcome(trial, expected)
    trajectory = trajectory_metrics(trial)
    safety = grade_safety(trial)
    findings = []

    if not outcome['passed']:
        findings.append('OUTCOME_FAILURE')
    if trajectory['repeated_identical_calls'] > 0:
        findings.append('INEFFICIENT_REPETITION')
    if not trajectory['verified_before_stop']:
        findings.append('MISSING_VERIFICATION')
    if not safety['safe']:
        findings.append('SAFETY_OR_AUTHORITY_FAILURE')
    return findings or ['CLEAN_PASS']

print(diagnose(trial, expected_state))

['INEFFICIENT_REPETITION']


# 12. Improve the system at the layer that failed

> Start with the cheapest intervention that addresses the observed failure. Training cannot repair a broken environment, misleading tool schema, or weak authorization boundary.


In [ ]:
_finish(improvement_figure())

In [8]:
before = {'success_rate': 0.62, 'avg_calls': 12.4, 'unsafe_rate': 0.030, 'avg_cost': 0.14}
after  = {'success_rate': 0.81, 'avg_calls': 7.1,  'unsafe_rate': 0.002, 'avg_cost': 0.09}

labels = {
    'success_rate': 'Success rate',
    'avg_calls': 'Average tool calls',
    'unsafe_rate': 'Unsafe-action rate',
    'avg_cost': 'Average cost ($)',
}

print(f"{'Metric':<24} {'Before':>10} {'After':>10} {'Delta':>10}")
print('-' * 58)
for key, label in labels.items():
    delta = after[key] - before[key]
    print(f'{label:<24} {before[key]:>10.3f} {after[key]:>10.3f} {delta:>+10.3f}')

print('Always compare on held-out capability and regression suites.')

Metric                       Before      After      Delta
----------------------------------------------------------
Success rate                  0.620      0.810     +0.190
Average tool calls           12.400      7.100     -5.300
Unsafe-action rate            0.030      0.002     -0.028
Average cost ($)              0.140      0.090     -0.050
Always compare on held-out capability and regression suites.


# 13. SFT, RL, and Jev solve different problems

| Approach | Input | What it changes | Best fit |
|---|---|---|---|
| **SFT** | High-quality demonstrations | Teaches imitation of desired responses and tool patterns | Stable, demonstrable behavior |
| **RL / RFT** | Environment plus robust reward | Learns policies that maximize measured outcomes | Verifiable success with room to explore |
| **Jev** | Current state plus closed choices | Makes a typed inference-time routing decision | Separating action selection from generation |

Jev is an inference-time architecture option—not a training method or an evaluation method. Measure **end-to-end task success**, argument correctness, abstention, latency, cost, and safety—not only top-1 tool-choice accuracy.


In [ ]:
_finish(cards_figure(
    [
        ('SFT', 'Imitate high-quality demonstrations'),
        ('RL / RFT', 'Optimize against a robust reward'),
        ('JEV', 'Choose among closed options at inference time'),
    ],
    [BLUE, VIOLET, GREEN],
    columns=3,
    height=350,
))

In [9]:
# Illustrative request shape; an external API call is intentionally omitted.
jev_request = {
    'state': {
        'goal': trial['goal'],
        'completed_steps': ['search'],
        'last_observation': 'Relevant evidence retrieved',
    },
    'question': {
        'type': 'choice',
        'instructions': 'Select the most appropriate next action.',
        'options': [
            'search', 'save_report', 'verify',
            'ask_user', 'respond_without_tool', 'stop',
        ],
    },
}

pprint(jev_request, sort_dicts=False)
print('The selected tool still needs valid arguments, execution, observation handling, and stopping logic.')

{'state': {'goal': 'Produce a verified result without exceeding authority.',
           'completed_steps': ['search'],
           'last_observation': 'Relevant evidence retrieved'},
 'question': {'type': 'choice',
              'instructions': 'Select the most appropriate next action.',
              'options': ['search',
                          'save_report',
                          'verify',
                          'ask_user',
                          'respond_without_tool',
                          'stop']}}
The selected tool still needs valid arguments, execution, observation handling, and stopping logic.


# 14. The continuous evaluation loop

Maintain separate suites:

- **Capability evals:** hard tasks that reveal the next improvement frontier
- **Regression evals:** previously solved behavior that should remain near-perfect
- **Safety evals:** invariants and adversarial cases that should never regress
- **Production monitoring:** drift, new failure modes, user feedback, and incident review


In [ ]:
_finish(cycle_figure())

# 15. Final checklist

- [ ] Did the agent achieve the actual goal?
- [ ] Does the environment state confirm the claim?
- [ ] Was the trajectory correct and reasonably efficient?
- [ ] Did the agent remain within its authority?
- [ ] Can it succeed consistently across repeated trials?
- [ ] Can we identify why it failed—not only that it failed?
- [ ] Did the proposed improvement pass held-out and regression tests?

## Closing idea

> **Evaluate outcomes. Inspect trajectories. Test boundaries. Measure distributions. Improve the layer that failed.**


---

## Further reading

- [Anthropic — Demystifying evals for AI agents](https://www.anthropic.com/research/demystifying-evals-for-ai-agents)
- [OpenAI — Evaluate agent workflows](https://developers.openai.com/api/docs/guides/agent-evals)
- [OpenAI — Trace grading](https://developers.openai.com/api/docs/guides/trace-grading)
- [TRAJECT-Bench](https://openreview.net/pdf?id=TZWnWvsQ0X)
- [AgentDojo](https://arxiv.org/abs/2406.13352)
- [NIST — Agent hijacking evaluations](https://www.nist.gov/news-events/news/2025/01/technical-blog-strengthening-ai-agent-hijacking-evaluations)
- [OpenAI — Reinforcement fine-tuning](https://developers.openai.com/api/docs/guides/reinforcement-fine-tuning)
- [TypeSafe AI — Jev](https://typesafe.ai/)

Visuals use Plotly's Colab renderer; NetworkX supplies graph structure and layouts.
